# LAB DAY 19: GraphRAG Demo

Notebook này chạy từng bước của pipeline: đọc corpus, trích xuất triples, xây dựng đồ thị, trực quan hóa, và kiểm tra truy vấn.

In [1]:
from pathlib import Path
import json
import sys

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from entity_extraction import EntityExtractor
from graph_builder import GraphBuilder
from query_engine import QueryEngine
CORPUS_PATH = PROJECT_ROOT / 'data' / 'raw' / 'tech_companies.txt'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'triples.json'
GRAPH_PATH = PROJECT_ROOT / 'outputs' / 'graphs' / 'knowledge_graph.png'
RESULTS_PATH = PROJECT_ROOT / 'outputs' / 'results' / 'benchmark_results.json'

print('Notebook ready')

Notebook ready


## 1. Load corpus

In [2]:
corpus_text = CORPUS_PATH.read_text(encoding='utf-8')
paragraphs = [p.strip() for p in corpus_text.split('\n\n') if p.strip()]
print('Paragraphs:', len(paragraphs))
print(paragraphs[0][:180])

Paragraphs: 16
OpenAI was founded in December 2015 by Sam Altman, Elon Musk, Greg Brockman, Ilya Sutskever, John Schulman, and Wojciech Zaremba. The founders wanted to build safe general-purpose 


## 2. Extract triples

In [3]:
extractor = EntityExtractor()
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
triples = extractor.extract_from_corpus(str(CORPUS_PATH), str(PROCESSED_PATH))
print('Triples extracted:', len(triples))
print(triples[:10])

[EntityExtractor] OpenAI extraction failed, falling back to rules: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************sUYA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
Extracted 7 triples
Extracted 1 triples
Extracted 4 triples
Extracted 2 triples
Extracted 3 triples
Extracted 2 triples
Extracted 7 triples
Extracted 1 triples
Extracted 2 triples
Extracted 2 triples
Extracted 4 triples
Extracted 2 triples
Extracted 3 triples
Extracted 1 triples
Extracted 4 triples
Extracted 2 triples
Triples extracted: 47
[('OpenAI', 'FOUNDED_BY', 'Sam Altman'), ('OpenAI', 'FOUNDED_BY', 'Elon Musk'), ('OpenAI', 'FOUNDED_BY', 'Greg Brockman'), ('OpenAI', 'FOUNDED_BY', 'Ilya Sutskever'), ('OpenAI', 'FOUNDED_B

## 3. Build graph

In [4]:
builder = GraphBuilder()
builder.add_triples(triples)
stats = builder.get_stats()
stats

{'num_nodes': 52,
 'num_edges': 45,
 'nodes': ['OpenAI',
  'Sam Altman',
  'Elon Musk',
  'Greg Brockman',
  'Ilya Sutskever',
  'John Schulman',
  'Wojciech Zaremba',
  '2015',
  'San Francisco',
  'Google',
  'Larry Page',
  'Sergey Brin',
  '1998',
  'Alphabet Inc.',
  'Mountain View',
  'Microsoft',
  'Bill Gates',
  'Paul Allen',
  '1975',
  'Satya Nadella',
  'Redmond',
  'Meta',
  'Mark Zuckerberg',
  'Eduardo Saverin',
  'Andrew McCollum',
  'Dustin Moskovitz',
  'Chris Hughes',
  '2004',
  'Menlo Park',
  'Amazon',
  'Jeff Bezos',
  '1994',
  'Andy Jassy',
  'Seattle',
  'Apple',
  'Steve Jobs',
  'Steve Wozniak',
  'Ronald Wayne',
  '1976',
  'Tim Cook',
  'Cupertino',
  'Tesla',
  'Martin Eberhard',
  'Marc Tarpenning',
  '2003',
  'Austin',
  'NVIDIA',
  'Jensen Huang',
  'Chris Malachowsky',
  'Curtis Priem',
  '1993',
  'Santa Clara'],
 'edges': [('OpenAI', 'Sam Altman', 'FOUNDED_BY'),
  ('OpenAI', 'Elon Musk', 'FOUNDED_BY'),
  ('OpenAI', 'Greg Brockman', 'FOUNDED_BY'),
 

## 4. Visualize graph

In [5]:
builder.visualize(str(GRAPH_PATH))
print(GRAPH_PATH)

Graph saved to /Users/nhugiabach/Documents/Github/VinUniAssignment/assignments-main/lab_day19_2A2026248_NhuGiaBach/outputs/graphs/knowledge_graph.png
/Users/nhugiabach/Documents/Github/VinUniAssignment/assignments-main/lab_day19_2A2026248_NhuGiaBach/outputs/graphs/knowledge_graph.png


## 5. Query examples

In [6]:
engine = QueryEngine(builder.graph)
queries = [
    'Who founded OpenAI?',
    'Who is the CEO of Microsoft?',
    'Where is Apple headquartered?',
    'Who is the CEO of the company founded by Bill Gates?',
    'Who founded the company whose CEO is Satya Nadella?'
]
for q in queries:
    print('Q:', q)
    print('A:', engine.answer(q))
    print('---')

Q: Who founded OpenAI?
A: ✅ OpenAI was founded by: Sam Altman, Elon Musk, Greg Brockman, Ilya Sutskever, John Schulman, Wojciech Zaremba
---
Q: Who is the CEO of Microsoft?
A: ✅ The current CEO of Microsoft is Satya Nadella
---
Q: Where is Apple headquartered?
A: ✅ Apple is headquartered in Cupertino
---
Q: Who is the CEO of the company founded by Bill Gates?
A: ✅ The current CEO of Microsoft is Satya Nadella
---
Q: Who founded the company whose CEO is Satya Nadella?
A: ✅ Microsoft was founded by: Bill Gates, Paul Allen
---


## 6. Read benchmark summary

In [7]:
data = json.loads(RESULTS_PATH.read_text())
data['summary']

{'num_questions': 20,
 'graph_accuracy_percent': 100.0,
 'flat_accuracy_percent': 75.0}